# ISC data harmonization

Transforms Rijkswaterstaat (RWS) ISC export data into the Dutch output format
required by the Internationale Scheldecommissie (ISC).

See `04_harmonize_isc_data_README.md` for full documentation.

## Configuration

In [58]:
import sys
from importlib import reload
from pathlib import Path

import pandas as pd

# Ensure the scripts folder is on sys.path (works from repo root or scripts/)
_scripts_dir = None
for candidate in (Path.cwd(), Path.cwd() / "scripts"):
    if (candidate / "isc_harmonization.py").exists():
        _scripts_dir = candidate
        break
if _scripts_dir is None:
    raise ImportError("Could not find isc_harmonization.py. Run the notebook from the repo or scripts folder.")
if str(_scripts_dir) not in sys.path:
    sys.path.insert(0, str(_scripts_dir))

import isc_harmonization

reload(isc_harmonization)  # pick up changes without restarting the kernel

from isc_harmonization import (
    add_fraction_labels_from_mapping,
    add_parameter_ids_from_mapping,
    add_station_ids_from_mapping,
    aggregate_compound_parameters,
    apply_isc_measurement_filters,
    build_harmonized_output,
    combine_and_finalize_output,
    export_final_output,
    format_result_values,
    get_repo_root,
    keep_lowest_aquocode_per_case,
    load_and_filter_chlorophyll_data,
    print_loaded_inputs,
    select_output_columns,
    sort_by_station_parameter_date,
    split_measured_and_not_measured_parameters,
)

TARGET_YEAR = 2024

REPO_ROOT = get_repo_root()
DATA_DIR = REPO_ROOT / "voorbeeld" / "isc_2023-2025"
MAPPING_DIR = REPO_ROOT / "mappings"

RAW_DATA_PATH = DATA_DIR / f"ISC_{TARGET_YEAR}.xlsx"
CHLOROPHYLL_PATH = DATA_DIR / "SCHAARVODDL + SASVGT_CHLfa_2023-2025.xlsx"
OUTPUT_BASE_PATH = DATA_DIR / f"ISC_{TARGET_YEAR}_harmonized"

## 1. Load inputs

In [59]:
raw_measurements = pd.read_excel(RAW_DATA_PATH, sheet_name=str(TARGET_YEAR))
location_mapping = pd.read_excel(MAPPING_DIR / "locations-mapped.xlsx")
parameter_mapping = pd.read_excel(
    MAPPING_DIR / "parameter_mapping_final.xlsx",
    sheet_name="mapping",
)
fraction_mapping = pd.read_excel(MAPPING_DIR / "hoedanigheid_mapped.xlsx")

print_loaded_inputs(
    raw_measurements,
    location_mapping,
    parameter_mapping,
    fraction_mapping,
)

measured_mapping, not_measured_mapping = split_measured_and_not_measured_parameters(
    parameter_mapping
)


STEP 0: Load input files
  > Raw measurements: 33,403 rows
  > Location mapping: 7 rows
  > Parameter mapping: 129 rows
  > Fraction mapping: 18 rows

STEP 1: Split parameter mapping
  Mapping table: 129 rows total
  > Measured combinations (used in pipeline): 101
  > Not measured (NM, appended at end): 21
  > Excluded (reported = S): 7


## 2. Map, filter, and harmonize measurements

In [60]:
measurements = add_parameter_ids_from_mapping(raw_measurements, measured_mapping)
measurements = add_station_ids_from_mapping(measurements, location_mapping)
measurements = add_fraction_labels_from_mapping(measurements, fraction_mapping)

filtered_measurements = apply_isc_measurement_filters(measurements)
filtered_measurements, removed_duplicate_aquocodes = keep_lowest_aquocode_per_case(
    filtered_measurements
)

harmonized = build_harmonized_output(filtered_measurements)


STEP 2: Map parameter combinations
  Input: 33,403 rows | 529 combinations
  > 529 unique combinations in raw data
  Mapping entries available: 101 unique combinations
  Output: 9,284 rows | 101 combinations
  - 24,119 rows dropped (72.2%) - no matching combination in mapping
  > 24,119 rows had no combination match and were removed
  > All 101 mapped combinations appear in the dataset

STEP 3: Map locations
  Input: 9,284 rows | 101 combinations
  Location codes in mapping: 7
  Output: 9,284 rows | 101 combinations | 7 stations
  > No row change - location mapping does not drop rows
  > All rows matched to a station ID

STEP 4: Map fractions
  Input: 9,284 rows | 101 combinations | 7 stations
  Fraction codes in mapping: 18
  Output: 9,284 rows | 101 combinations | 7 stations | 2 fractions
  > No row change - fraction mapping does not drop rows
  > All rows matched to a fraction label

STEP 5: Filter measurements
  Input: 9,284 rows | 101 combinations | 7 stations | 2 fractions
  > 1

## 3. Compress compound parameters

In [ ]:
harmonized_compressed, incomplete_1319, lq_conflicts_1319 = aggregate_compound_parameters(
    harmonized,
    source_param_ids=["1551", "1339", "1340"],
    source_ops=["+", "-", "-"],
    target_param_id="1319",
    remove_source_rows=False,
 )

harmonized_compressed, incomplete_1774, lq_conflicts_1774 = aggregate_compound_parameters(
    harmonized_compressed,
    source_param_ids=["1283", "1629", "1630"],
    source_ops=["+", "+", "+"],
    target_param_id="1774",
    remove_source_rows=False,
 )

harmonized_compressed, incomplete_5534, lq_conflicts_5534 = aggregate_compound_parameters(
    harmonized_compressed,
    source_param_ids=["1103", "1181", "1173", "1207"],
    source_ops=["+", "+", "+", "+"],
    target_param_id="5534",
    remove_source_rows=False,
 )

harmonized_compressed, incomplete_5537, lq_conflicts_5537 = aggregate_compound_parameters(
    harmonized_compressed,
    source_param_ids=["1200", "1201", "1202", "1203"],
    source_ops=["+", "+", "+", "+"],
    target_param_id="5537",
    remove_source_rows=False,
 )

harmonized_compressed, incomplete_6561, lq_conflicts_6561 = aggregate_compound_parameters(
    harmonized_compressed,
    source_param_ids=["6561a", "6561b"],
    source_ops=["+", "+"],
    target_param_id="6561",
    remove_source_rows=True,
 )

harmonized_compressed, incomplete_7706, lq_conflicts_7706 = aggregate_compound_parameters(
    harmonized_compressed,
    source_param_ids=["1197", "1198"],
    source_ops=["+", "+"],
    target_param_id="7706",
    remove_source_rows=False,
 )


Aggregate compound combinations -> 1774
  Source combinations (by ISC target): ['1283', '1629', '1630']
  Remove source rows after aggregation: False
  Input: 6,861 rows | 101 combinations | 5 stations | 2 fractions
  > 189 source rows matched for aggregation
  > All groups contain the full set of source combinations
  > LQ symbols are consistent within all groups
  > 63 aggregated rows created from 189 source rows
  > Source rows kept (aggregated rows will be appended)
  Output: 6,924 rows | 102 combinations | 5 stations | 2 fractions
  + 63 rows added - aggregated source combinations into target 1774

Aggregate compound combinations -> 5534
  Source combinations (by ISC target): ['1103', '1181', '1173', '1207']
  Remove source rows after aggregation: False
  Input: 6,924 rows | 102 combinations | 5 stations | 2 fractions
  > 260 source rows matched for aggregation
  > All groups contain the full set of source combinations
  > LQ symbols are consistent within all groups
  > 65 aggreg

## 4. Format and select output columns

In [62]:
harmonized_for_report = sort_by_station_parameter_date(harmonized_compressed)

harmonized_output = select_output_columns(
    format_result_values(harmonized_for_report)
)

harmonized_output.head()


STEP 8: Sort output
  Input: 6,937 rows | 100 combinations | 5 stations | 2 fractions
  > Sorting by station -> combination -> date

STEP 9: Format result values
  Input: 6,937 rows | 100 combinations | 5 stations | 2 fractions
  > 6,937 numeric results formatted to 4 decimals
  > 45 results set to 'NV' (aquocode 99)
  Output: 6,937 rows | 100 combinations | 5 stations | 2 fractions
  > No row change - formatting only, no rows dropped

STEP 10: Select output columns
  Input: 6,937 rows | 100 combinations | 5 stations | 2 fractions
  > Keeping 7 Dutch output columns
  Output: 6,937 rows | 5 stations | 2 fractions
  > No row change - column selection only, no rows dropped


,Unieke identiticatie meetpunt,Datum staalname,Geanalyseerde fractie,Unieke identificatie gemeten parameter,Aanpak kwantificeringsgrens,Resultaat,Unieke identificatie van de eenheid
0,NL89_SCHAARVODDL,16/01/2024,EB,1082,=,0.0161,µg/L
1,NL89_SCHAARVODDL,15/02/2024,EB,1082,=,0.0087,µg/L
2,NL89_SCHAARVODDL,12/03/2024,EB,1082,=,0.0068,µg/L
3,NL89_SCHAARVODDL,10/04/2024,EB,1082,=,0.0169,µg/L
4,NL89_SCHAARVODDL,13/05/2024,EB,1082,=,0.0182,µg/L


## 5. Append chlorophyll-a and not-measured (NM) rows

In [63]:
chlorophyll_target_year = load_and_filter_chlorophyll_data(
    CHLOROPHYLL_PATH,
    TARGET_YEAR,
)

final_output = combine_and_finalize_output(
    harmonized_output,
    chlorophyll_target_year,
    not_measured_mapping,
    measured_data_with_combinations=harmonized_for_report,
    measured_mapping=measured_mapping,
)


STEP 11: Load chlorophyll-a data
  > SCHAARVODDL rows loaded: 83
  > SASVGT rows loaded: 43
  > Total before year filter: 126
  > Rows for 2024: 39
  Output: 39 rows | 2 stations | 1 fractions

STEP 13: Combine harmonized data with chlorophyll
  Harmonized measurements: 6,937 rows | 5 stations | 2 fractions
  Chlorophyll: 39 rows | 2 stations | 1 fractions
  Combined: 6,976 rows | 6 stations | 2 fractions
  + 39 rows added - chlorophyll rows appended

STEP 14: Create not-measured (NM) rows
  > 20 NM rows created from parameter mapping
  > Each row has Resultaat = 'NM' with empty station, fraction, and date

STEP 15: Final dataset ready
  Final dataset: 6,996 rows | 7 stations | 3 fractions
  > Measured + chlorophyll rows: 6,976
  > NM rows appended: 20

STEP 16: Station combination report
  Expected measured combinations in mapping: 101
  Globally not measured (NM): 21 mapping row(s)
  > 20 unique parameter+unit pair(s) in NM export (1 duplicate mapping row(s) merged)
    - ISC parame

## 6. Export

In [66]:
export_final_output(
    final_output,
    OUTPUT_BASE_PATH,
    sheet_name=str(TARGET_YEAR),
)


STEP 17: Export output files
  > Saved CSV: C:\Ocean\Work\Projects\2026\ISC\Tools\Repositories\voorbeeld\isc_2023-2025\ISC_2024_harmonized.csv
  > Saved Excel: C:\Ocean\Work\Projects\2026\ISC\Tools\Repositories\voorbeeld\isc_2023-2025\ISC_2024_harmonized.xlsx
  > All output columns exported as text (supports numbers and labels like NV/NM)


(WindowsPath('C:/Ocean/Work/Projects/2026/ISC/Tools/Repositories/voorbeeld/isc_2023-2025/ISC_2024_harmonized.csv'),
 WindowsPath('C:/Ocean/Work/Projects/2026/ISC/Tools/Repositories/voorbeeld/isc_2023-2025/ISC_2024_harmonized.xlsx'))